In [2]:
import speech_recognition as sr # pip install SpeechRecognition

# 1. 初始化辨識器
r = sr.Recognizer()

# 2. 開啟麥克風錄音
with sr.Microphone() as source:
    print("請開始說話...")   
    r.adjust_for_ambient_noise(source, duration=1)#duration=1校準環境噪音1秒，先不要說話
    
    # 聽取聲音
    audio = r.listen(source)

print("錄音結束，正在辨識中...")

# 3. 直接辨識並印出結果 (不使用 try 區塊)
text = r.recognize_google(audio, language="zh-TW")
print(f"你說的是：{text}")

請開始說話...
錄音結束，正在辨識中...
你說的是：嗨你好嗨你好


In [3]:
import speech_recognition as sr

r = sr.Recognizer()

# --- 核心參數設定 ---
r.pause_threshold = 1.8         # 停頓 1.8 秒才斷句，確保長句完整
r.dynamic_energy_threshold = True # 自動偵測環境噪音
r.energy_threshold = 400        # 基礎靈敏度

with sr.Microphone() as source:
    print("\n[系統] 語音模式啟動。請直接說話，說完停頓一下即可。請開始說話...")
    r.adjust_for_ambient_noise(source, duration=1)

    while True:
        print("\n(正在聽...)", end="", flush=True)
        
        # 錄音
        audio = r.listen(source)
        
        print("\r[正在辨識...]", end="", flush=True)
        
        # 辨識文字 (使用 Google 雲端)
        text = r.recognize_google(audio, language="zh-TW")
        
        print(f'{text}')
        
        # 結束指令判斷
        if "結束" in text or "停止" in text or "退出" in text:
            print("再見！系統關閉中...")
            break


[系統] 語音模式啟動。請直接說話，說完停頓一下即可。請開始說話...

[正在辨識...]hi你好歡迎光臨石二鍋結束
再見！系統關閉中...


In [6]:
import speech_recognition as sr
from deep_translator import GoogleTranslator

# 1. 初始化辨識器
r = sr.Recognizer()

# --- 核心參數調整 (解決圖片中話被切斷的問題) ---
# 增加等待時間到 2.5 秒，確保像「早餐因為我喜...」之後的字能被抓到
r.pause_threshold = 2.5        
r.dynamic_energy_threshold = True 
r.energy_threshold = 400       

# 2. 開啟麥克風並開始執行
with sr.Microphone() as source:
    print(">>> 正在校準背景雜音 (請保持安靜 1 秒)...")
    r.adjust_for_ambient_noise(source, duration=1.0)
    
    print("\n=== 翻譯系統已啟動 ===")
    print("請開始說話，我會翻譯成英文與日文。")
    print("(提示：說完請停頓約 2 秒；說「結束」可關閉程式)")
    print("="*40)

    while True:
        try:
            # 顯示狀態，使用 \r 讓畫面保持整潔
            print("(正在聽...)", end="\r", flush=True)
            
            # 錄取語音：phrase_time_limit=15 確保長句子最長錄 15 秒即強制處理
            audio = r.listen(source, phrase_time_limit=15)
            
            print("[處理中...]       ", end="\r", flush=True)
            
            # 3. 語音轉文字 (中文)
            text_zh = r.recognize_google(audio, language="zh-TW")
            
            if not text_zh.strip():
                continue

            # 4. 執行多國翻譯
            text_en = GoogleTranslator(source='zh-TW', target='en').translate(text_zh)
            text_ja = GoogleTranslator(source='zh-TW', target='ja').translate(text_zh)
            
            # 5. 顯示結果
            print(" " * 40, end="\r") # 清除 [處理中] 字樣
            print(f"【中文】: {text_zh}")
            print(f"【English】: {text_en}")
            print(f"【日本語】: {text_ja}")
            print("-" * 40)
            
            # 結束指令判定
            if any(k in text_zh for k in ["結束", "停止", "退出"]):
                print("\n[系統] 收到指令，程式結束。")
                break
                
        except sr.UnknownValueError:
            # 當 Google 聽不到清晰字句（如純雜音）時，自動忽略並繼續
            continue 
        except Exception as e:
            # 捕捉其他可能的網路或系統錯誤
            print(f"\n[系統提示] 發生錯誤: {e}")
            break

>>> 正在校準背景雜音 (請保持安靜 1 秒)...

=== 翻譯系統已啟動 ===
請開始說話，我會翻譯成英文與日文。
(提示：說完請停頓約 2 秒；說「結束」可關閉程式)
【中文】: 很高興見到                             
【English】: nice to see
【日本語】: 見てよかった
----------------------------------------
【中文】: 很高興見                              
【English】: nice to see
【日本語】: 見てよかった
----------------------------------------
【中文】: 很高興見到                             
【English】: nice to see
【日本語】: 見てよかった
----------------------------------------
【中文】: 很高興見                              
【English】: nice to see
【日本語】: 見てよかった
----------------------------------------
【中文】: 大家看一下你很棒                          
【English】: Everyone, take a look, you are great
【日本語】: 皆さん、見てください、あなたは素晴らしいです
----------------------------------------
【中文】: 你很棒的畫面                            
【English】: Your great picture
【日本語】: あなたの素晴らしい写真
----------------------------------------
【中文】: 現在好像準                             
【English】: It seems accurate now
【日本語】: 今では正確なようです
----------------------------------------
【中文】: 你好像每